In [3]:
!pip install --upgrade pip
!pip install torch torchvision torchaudio
!pip install pandas numpy scikit-learn Pillow matplotlib seaborn

  Using cached torch-2.11.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (5.5 kB)
  Using cached torchaudio-2.11.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (6.9 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached numpy-2.4.4-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached pillow-12.2.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.w

In [5]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

BASE_DIR = '/Users/toxa/Desktop/GUESER/dataset'
TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'train_images')
CSV_PATH = os.path.join(BASE_DIR, 'train_solution.csv')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Работаем на: {device}")

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SEBlock(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += self.shortcut(x)
        return F.relu(out)

class AuraDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.in_channels = 32
        self.conv1 = nn.Conv2d(3, 32, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        self.layer1 = self._make_layer(64, 2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(512, 1)
        )
    def _make_layer(self, out_channels, blocks, stride):
        layers = [ResidualBlock(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return self.classifier(x)

class DeepFakeDataset(Dataset):
    def __init__(self, root_dir, dataframe, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        files_in_folder = set(os.listdir(root_dir))
        def check_exists(img_id):
            name = str(img_id)
            return (name + '.jpg' in files_in_folder) or (name in files_in_folder)
        mask = dataframe['Id'].apply(check_exists)
        self.df = dataframe[mask].reset_index(drop=True)
        print(f"Найдено картинок: {len(self.df)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx, 0])
        img_name = img_id if img_id.endswith('.jpg') else img_id + '.jpg'
        img_path = os.path.join(self.root_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        label = torch.tensor(self.df.iloc[idx, 1], dtype=torch.float32)
        if self.transform:
            image = self.transform(image)
        return image, label

train_trans = T.Compose([
    T.Resize((256, 256)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Не найден CSV файл по пути: {CSV_PATH}")

raw_df = pd.read_csv(CSV_PATH, header=None, names=['Id', 'target'])
train_data, val_data = train_test_split(raw_df, test_size=0.1, stratify=raw_df['target'], random_state=42)

train_loader = DataLoader(DeepFakeDataset(TRAIN_IMG_DIR, train_data, train_trans), batch_size=32, shuffle=True)
val_loader = DataLoader(DeepFakeDataset(TRAIN_IMG_DIR, val_data, train_trans), batch_size=32)

model = AuraDetector().to(device)
pos_weight = torch.tensor([len(raw_df[raw_df['target']==0]) / len(raw_df[raw_df['target']==1])]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for epoch in range(10):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 100 == 0:
            print(f"Эпоха [{epoch+1}/10] | Батч [{i}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    model.eval()
    preds_list, labels_list = [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            outputs = torch.sigmoid(model(imgs.to(device)))
            preds_list.extend((outputs > 0.5).cpu().numpy())
            labels_list.extend(lbls.numpy())
    
    score = f1_score(labels_list, preds_list)
    print(f"🌟 Эпоха {epoch+1}: F1-Score = {score:.4f} | Avg Loss = {running_loss/len(train_loader):.4f}")
    torch.save(model.state_dict(), f"aura_v1_epoch_{epoch+1}.pth")

Работаем на: mps
Найдено картинок: 45000
Найдено картинок: 5000
Эпоха [1/10] | Батч [0/1407] | Loss: 1.2758
Эпоха [1/10] | Батч [100/1407] | Loss: 1.2232
Эпоха [1/10] | Батч [200/1407] | Loss: 1.0127
Эпоха [1/10] | Батч [300/1407] | Loss: 1.2519
Эпоха [1/10] | Батч [400/1407] | Loss: 0.6665
Эпоха [1/10] | Батч [500/1407] | Loss: 1.1342
Эпоха [1/10] | Батч [600/1407] | Loss: 0.9831
Эпоха [1/10] | Батч [700/1407] | Loss: 0.7792
Эпоха [1/10] | Батч [800/1407] | Loss: 0.5553
Эпоха [1/10] | Батч [900/1407] | Loss: 0.5563
Эпоха [1/10] | Батч [1000/1407] | Loss: 0.8286
Эпоха [1/10] | Батч [1100/1407] | Loss: 0.6509
Эпоха [1/10] | Батч [1200/1407] | Loss: 0.6001
Эпоха [1/10] | Батч [1300/1407] | Loss: 0.5010
Эпоха [1/10] | Батч [1400/1407] | Loss: 1.4063
🌟 Эпоха 1: F1-Score = 0.4509 | Avg Loss = 0.9552
Эпоха [2/10] | Батч [0/1407] | Loss: 0.6942
Эпоха [2/10] | Батч [100/1407] | Loss: 0.9813
Эпоха [2/10] | Батч [200/1407] | Loss: 0.6122
Эпоха [2/10] | Батч [300/1407] | Loss: 0.9469
Эпоха [2/10]